In [4]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [5]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year in (2023,2024)
    AND ud.month = 1
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")
# Convert query results into dataframe
df = regional_uniques.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

# Initialize an empty DataFrame to hold all results
final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis_mobile.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'formula_result_percentage_regional', 'simple_calculation_global', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy.csv", index = False, header=False)
    
# Show df
final_df

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,simple_calculation_regional,formula_result_percentage_regional,simple_calculation_global,formula_result_percentage_global
134,2024-01,Russia,Central & Eastern Europe & Central Asia,62822014.0,71399762.0,8577748.0,37.412052,10.348380,6.593707,6.699275
115,2024-01,Bulgaria,Central & Eastern Europe & Central Asia,9653948.0,4381385.0,5272563.0,22.996409,50.638825,4.053015,4.273981
131,2024-01,Poland,Central & Eastern Europe & Central Asia,23274029.0,25193173.0,1919144.0,8.370392,3.789121,1.475244,1.615945
140,2024-01,Ukraine,Central & Eastern Europe & Central Asia,17752751.0,18962421.0,1209670.0,5.276004,1.397707,0.929872,1.036782
139,2024-01,Turkey,Central & Eastern Europe & Central Asia,25512372.0,26490801.0,978429.0,4.267441,4.112798,0.752118,0.869721
...,...,...,...,...,...,...,...,...,...,...
195,2024-01,Zambia,Sub-Saharan Africa,450155.0,450692.0,537.0,0.006533,2.638350,0.000413,0.002040
189,2024-01,French Southern and Antarctic Lands,Sub-Saharan Africa,6.0,39.0,33.0,0.000401,0.000759,0.000025,0.000027
155,2024-01,Western Sahara,Sub-Saharan Africa,141.0,127.0,14.0,0.000170,0.001177,0.000011,0.000011
162,2024-01,Equatorial Guinea,Sub-Saharan Africa,35593.0,35585.0,8.0,0.000097,0.211957,0.000006,0.000120


## Mobile enwiki breakdown

In [6]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.year in (2023,2024)
    AND ud.month = 1
    and ud.domain like '%.m.wikipedia%'
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")

# Convert query results into dataframe
df = regional_uniques_mobile.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis_mobile.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'formula_result_percentage_regional', 'simple_calculation_global', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy_mobile.csv", index = False, header = False)
    
# Show df
final_df

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,simple_calculation_regional,formula_result_percentage_regional,simple_calculation_global,formula_result_percentage_global
134,2024-01,Russia,Central & Eastern Europe & Central Asia,50286984.0,53550686.0,3263702.0,24.769226,28.060740,2.821915,7.614303
115,2024-01,Bulgaria,Central & Eastern Europe & Central Asia,5206666.0,4093620.0,1113046.0,8.447244,9.555263,0.962380,1.041825
133,2024-01,Serbia,Central & Eastern Europe & Central Asia,7172406.0,6158476.0,1013930.0,7.695023,7.821598,0.876681,0.762702
141,2024-01,Uzbekistan,Central & Eastern Europe & Central Asia,4637255.0,3702280.0,934975.0,7.095809,7.992393,0.808413,0.855904
139,2024-01,Turkey,Central & Eastern Europe & Central Asia,25616841.0,24701667.0,915174.0,6.945534,0.589701,0.791293,0.760772
...,...,...,...,...,...,...,...,...,...,...
166,2024-01,Liberia,Sub-Saharan Africa,125170.0,125422.0,252.0,0.008001,0.202898,0.000218,0.009812
152,2024-01,Cameroon,Sub-Saharan Africa,898757.0,898615.0,142.0,0.004509,1.337070,0.000123,0.067886
189,2024-01,French Southern and Antarctic Lands,Sub-Saharan Africa,5.0,22.0,17.0,0.000540,0.000740,0.000015,0.000022
155,2024-01,Western Sahara,Sub-Saharan Africa,137.0,126.0,11.0,0.000349,0.000263,0.000010,0.000004
